![alt text](./Cerny_logo_1.jpg)

# Analysis of Cerny ventilation recordings

#### Processing clinical details

This notebook imports and processes clinical data and exports it into a pickle archive.

- Clinical data available in **850 cases**
- Only keep clinical data for cases where ventilation recordings (>5 minutes) are also available: **819 cases**

The data processed and analysed in this Notebook were collected by the **Neonatal Emergency and Transport Service of the Peter Cerny Foundation**, Budapest, Hungary

**Author: Dr Gusztav Belteki**

### 1. Import the required libraries and set options

In [1]:
import IPython
import pandas as pd
import numpy as np
import scipy as sp
import matplotlib
import matplotlib.pyplot as plt

import os
import sys
import re
import pickle

from scipy import stats
from pandas import Series, DataFrame
from datetime import datetime, timedelta

%matplotlib inline
matplotlib.style.use('classic')
matplotlib.rcParams['figure.facecolor'] = 'w'

pd.set_option('display.max_rows', 250)
pd.set_option('display.max_columns', 250)
pd.set_option('mode.chained_assignment', None)

# This is to turn off a warning message which is given when read_Excel() imports '.xlsx' files
import warnings
warnings.simplefilter("ignore")

In [2]:
print("Python version: {}".format(sys.version))
print("pandas version: {}".format(pd.__version__))
print("matplotlib version: {}".format(matplotlib.__version__))
print("NumPy version: {}".format(np.__version__))
print("SciPy version: {}".format(sp.__version__))
print("IPython version: {}".format(IPython.__version__))

Python version: 3.12.11 | packaged by conda-forge | (main, Jun  4 2025, 14:38:53) [Clang 18.1.8 ]
pandas version: 2.3.2
matplotlib version: 3.10.6
NumPy version: 1.26.4
SciPy version: 1.16.2
IPython version: 9.5.0


### 2. List and set the working directory and the directory to write out data

In [244]:
# Name of the external hard drive
DRIVE = 'GB_1'

# Directory on external drive to read the clinical data from
DIR_READ = os.path.join(os.sep, 'Volumes', DRIVE, 'ventilator_data', 'Fabian', 'fabian_patient_data_all')

# Path to project folder containing ventilation research results
PATH = os.path.join(os.sep, 'Users', 'guszti', 'Library', 'Mobile Documents', 'com~apple~CloudDocs', 
                            'Documents', 'Research', 'Ventilation')

# Folder to export the result of analysis
DIR_WRITE = os.path.join(PATH, 'ventilation_fabian', 'Analyses',  'analysis_1_1100')
os.makedirs(DIR_WRITE, exist_ok = True)

# Folder on external drive to export data to and to import processed data exported by other Notebooks
DATA_DUMP = os.path.join(os.sep, 'Volumes', DRIVE, 'data_dump', 'fabian',)
os.makedirs(DATA_DUMP, exist_ok = True)

DIR_READ, DIR_WRITE, DATA_DUMP

('/Volumes/GB_1/ventilator_data/Fabian/fabian_patient_data_all',
 '/Users/guszti/Library/Mobile Documents/com~apple~CloudDocs/Documents/Research/Ventilation/ventilation_fabian/Analyses/analysis_1_1100',
 '/Volumes/GB_1/data_dump/fabian')

### 3. Import ventilation data

This is needed to know the beginning and the end of the recordings

In [6]:
with open(os.path.join(DATA_DUMP, 'data_pars_1_150.pickle'), 'rb') as handle:
    data_pars_1_150 = pickle.load(handle)
with open(os.path.join(DATA_DUMP, 'data_pars_151_300.pickle'), 'rb') as handle:
    data_pars_151_300 = pickle.load(handle)
with open(os.path.join(DATA_DUMP, 'data_pars_301_450.pickle'), 'rb') as handle:
    data_pars_301_450 = pickle.load(handle) 
with open(os.path.join(DATA_DUMP, 'data_pars_451_600.pickle'), 'rb') as handle:
    data_pars_451_600 = pickle.load(handle)  
with open(os.path.join(DATA_DUMP, 'data_pars_601_750.pickle'), 'rb') as handle:
    data_pars_601_750 = pickle.load(handle)    
with open(os.path.join(DATA_DUMP, 'data_pars_751_900.pickle'), 'rb') as handle:
    data_pars_751_900 = pickle.load(handle) 
with open(os.path.join(DATA_DUMP, 'data_pars_901_1050.pickle'), 'rb') as handle:
    data_pars_901_1050 = pickle.load(handle)  
with open(os.path.join(DATA_DUMP, 'data_pars_1051_1100.pickle'), 'rb') as handle:
    data_pars_1051_1100 = pickle.load(handle)
    
data_pars = {**data_pars_1_150, **data_pars_151_300, **data_pars_301_450, **data_pars_451_600, **data_pars_601_750, 
             **data_pars_751_900, **data_pars_901_1050, **data_pars_1051_1100}

In [7]:
len(data_pars)

914

### 4. Import clinical data

In [18]:
# import text files in a dictionary
clin_dict = {}
for fname in os.listdir(DIR_READ):
    if not fname.startswith('.'): # disregard hidden files
        fhandle = open(os.path.join(DIR_READ, fname), 'r', encoding = 'cp1252')
        clin_dict[fname[:-4]] = fhandle.read() # use the filenames without the .txt extension as keys
        fhandle.close()

In [20]:
# split the clinical data into a list
for key in sorted(clin_dict.keys()):
    clin_dict[key] = clin_dict[key].split('\n')[:-1]

In [22]:
# Create an inner dictionary for the different clinical data
for key, value in sorted(clin_dict.items()):
    temp_dict = {}
    for item in value:
        td_key, *td_value = item.split(':')
        td_key = td_key.strip()
        temp_dict[td_key] = ''.join(td_value)[1:]
    clin_dict[key] = temp_dict

In [24]:
# Create a DataFrame from the dictionary of dictionaries
clin_df = DataFrame(clin_dict).T
clin_df.index.name = 'Recording_ID'
clin_df.sort_index(inplace = True)
len(clin_df)

925

### 5. Drop cases which have no clinical data

In [26]:
clin_df.dropna(axis = 0, how = 'all', inplace = True)
len(clin_df)

850

In [246]:
writer = pd.ExcelWriter(os.path.join(DIR_WRITE, 'clinical_data_all_1_1100_unfiltered.xlsx'))
clin_df.to_excel(writer, 'clin_df')
writer.close()

### 6. Drop cases for which there is no ventilation data

Ventilation recordings may have been excluded because they were two short (<15 mintes total) or aberrant

In [31]:
combined = sorted(set(list(clin_df.index)) & set(data_pars.keys()))
clin_df = clin_df.loc[combined]
len(clin_df)

819

### 7. Clean up clinical dataframe

In [34]:
# Curate the time of births of some recordings after manual inspection of case notes
clin_df.loc['AL000360']['Date of Birth'] = '20180906 0707'
clin_df.loc['AL000638']['Date of Birth'] = '20190814 1114'

In [36]:
# Change order of columns and create English names
clin_df = clin_df[['Esetlap id', 'Date of Birth', 'Gestation Age', 'Birth Weight', 'Actual Weight', 'Pathology', 'Start', 'End']]
clin_df.columns = ['Case ID', 'Date of Birth', 'Gestational Age', 'Birth Weight', 'Actual Weight', 'Pathology', 'Start', 'End']

In [38]:
clin_df['Gestational Age'] = clin_df['Gestational Age'].map(lambda x: int(x[:2]))
clin_df['Birth Weight'] = clin_df['Birth Weight'].map(lambda x: int(x[:-6]))
clin_df['Actual Weight'] = clin_df['Actual Weight'].str.strip(' grams')

In [40]:
actual_weight = []
for i in range(len(clin_df)):
    if clin_df.iloc[i]['Actual Weight'] == '':
        actual_weight.append(clin_df.iloc[i]['Birth Weight'])
    else:
        actual_weight.append(int(clin_df.iloc[i]['Actual Weight']))

clin_df['Weight'] = actual_weight

#### Start and end from ventilation data
This shows the time points when ventilator was turned on and off. At the beginning and the end of the recoridngs the baby was usually not attached to the ventilator. The ventilator recordings have been manually inspected and have been trimmed accordingly.

In [43]:
starts = {}; ends = {}
for rec in sorted(clin_df.index):
    try:
        starts[rec] = data_pars[rec].index[0]
    except KeyError:
        continue
        
    try:
        ends[rec] = data_pars[rec].index[-1]
    except KeyError:
        continue
        
start_end = DataFrame([starts, ends]).T
start_end.columns = ['Recording start', 'Recording end']

In [45]:
clin_df = pd.concat([clin_df, start_end], axis = 1, join = 'outer')
clin_df['Date of Birth'] = clin_df['Date of Birth'].map(lambda x: pd.to_datetime(x))
clin_df['Pathology'] = clin_df['Pathology'].map(lambda x: x.split(';')[:-1])
clin_df['Duration'] = clin_df['Recording end'] - clin_df['Recording start']
clin_df['Postnatal Age']   = clin_df['Recording end'] - clin_df['Date of Birth']

In [47]:
clin_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 819 entries, AL000003 to AL001086
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype          
---  ------           --------------  -----          
 0   Case ID          819 non-null    object         
 1   Date of Birth    819 non-null    datetime64[ns] 
 2   Gestational Age  819 non-null    int64          
 3   Birth Weight     819 non-null    int64          
 4   Actual Weight    819 non-null    object         
 5   Pathology        819 non-null    object         
 6   Start            819 non-null    object         
 7   End              819 non-null    object         
 8   Weight           819 non-null    int64          
 9   Recording start  819 non-null    datetime64[ns] 
 10  Recording end    819 non-null    datetime64[ns] 
 11  Duration         819 non-null    timedelta64[ns]
 12  Postnatal Age    819 non-null    timedelta64[ns]
dtypes: datetime64[ns](3), int64(3), object(5), timedelta64[ns](2)
memory usag

In [49]:
clin_df['Gestational Age'] = pd.to_timedelta((clin_df['Gestational Age']), unit='W', errors='raise')
clin_df['Corrected gestational Age'] = pd.to_timedelta((clin_df['Gestational Age']), unit='D', 
                                                       errors='raise') + clin_df['Postnatal Age']
clin_df['Gestational Age (weeks)'] = clin_df['Gestational Age'].apply(lambda x: x.total_seconds() / (60 * 60 * 24 * 7))
clin_df['Corrected gestational Age (weeks)'] = clin_df['Corrected gestational Age'].apply(lambda x: round(x.total_seconds() / (60 * 60 * 24 * 7), 1))

In [51]:
clin_df.sort_index(axis = 1).head()

,Actual Weight,Birth Weight,Case ID,Corrected gestational Age,Corrected gestational Age (weeks),Date of Birth,Duration,End,Gestational Age,Gestational Age (weeks),Pathology,Postnatal Age,Recording end,Recording start,Start,Weight
AL000003,,990,42543,196 days 02:09:04,28.0,2017-03-24 17:41:00,0 days 01:44:57,"Fri, 24 Mar 2017 195004 +0100",196 days,28.0,"[Neonat praemat gr s 28 (P07.3) , IRDS (P22....",0 days 02:09:04,2017-03-24 19:50:04,2017-03-24 18:05:07,"Fri, 24 Mar 2017 180507 +0100",990
AL000005,,3530,42552,259 days 02:07:19,37.0,2017-03-26 17:50:00,0 days 00:42:26,"Sun, 26 Mar 2017 195719 +0200",259 days,37.0,"[Neonat mat gr s 37 (P96.4) , Infectio (P39....",0 days 02:07:19,2017-03-26 19:57:19,2017-03-26 19:14:53,"Sun, 26 Mar 2017 191453 +0200",3530
AL000006,,1470,42554,217 days 01:18:07,31.0,2017-03-26 23:37:00,0 days 00:50:46,"Mon, 27 Mar 2017 005507 +0200",217 days,31.0,"[Neonat praemat gr s 31 (P07.3) , Gemini A (...",0 days 01:18:07,2017-03-27 00:55:07,2017-03-27 00:04:21,"Mon, 27 Mar 2017 000421 +0200",1470
AL000007,4800,3200,42578,339 days 02:48:57,48.4,2017-01-29 00:00:00,0 days 02:00:28,"Wed, 29 Mar 2017 024857 +0200",280 days,40.0,"[Exsiccatio (E86) , Légzési elégtelenség (P2...",59 days 02:48:57,2017-03-29 02:48:57,2017-03-29 00:48:29,"Wed, 29 Mar 2017 004829 +0200",4800
AL000008,,3230,42596,266 days 04:22:48,38.0,2017-03-29 13:20:00,0 days 01:58:22,"Wed, 29 Mar 2017 174248 +0200",266 days,38.0,"[Neonat mat gr s 38 (P96.4) , Intézeten kívü...",0 days 04:22:48,2017-03-29 17:42:48,2017-03-29 15:44:26,"Wed, 29 Mar 2017 154426 +0200",3230


### 8. EDA on clinical details

In [54]:
clin_df.describe()

,Date of Birth,Gestational Age,Birth Weight,Weight,Recording start,Recording end,Duration,Postnatal Age,Corrected gestational Age,Gestational Age (weeks),Corrected gestational Age (weeks)
count,819,819,819.000000,819.000000,819,819,819,819,819,819.000000,819.000000
mean,2019-02-14 23:58:38.534798592,242 days 16:00:00,2433.476190,2600.455433,2019-02-27 18:27:50.815628544,2019-02-27 19:43:09.452991232,0 days 01:15:18.637362637,12 days 19:44:30.918192918,255 days 11:44:30.918192916,34.666667,36.486691
min,2016-12-13 00:00:00,147 days 00:00:00,300.000000,300.000000,2017-03-24 18:05:07,2017-03-24 19:50:04,0 days 00:05:44,0 days 00:01:09,147 days 01:41:03,21.000000,21.000000
25%,2018-04-12 00:00:00,224 days 00:00:00,1690.000000,1840.000000,2018-04-14 20:56:54.500000,2018-04-14 22:05:06.500000,0 days 00:45:08,0 days 03:08:40,231 days 03:22:47.500000,32.000000,33.000000
50%,2019-04-17 00:00:00,252 days 00:00:00,2600.000000,2700.000000,2019-04-28 16:18:46,2019-04-28 18:40:14,0 days 01:05:40,0 days 07:03:47,259 days 02:10:51,36.000000,37.000000
75%,2020-02-07 05:34:30,266 days 00:00:00,3195.000000,3325.000000,2020-02-20 13:55:56.500000,2020-02-20 14:58:44.500000,0 days 01:36:41,6 days 06:45:28,273 days 20:01:59,38.000000,39.100000
max,2020-09-04 17:56:00,294 days 00:00:00,5400.000000,5920.000000,2020-09-21 14:30:58,2020-09-21 15:19:20,0 days 19:03:15,231 days 10:33:32,503 days 13:40:44,42.000000,71.900000
std,NaN,33 days 22:20:47.990023412,1021.977842,1039.097453,NaN,NaN,0 days 00:55:04.723952920,30 days 11:29:03.558109373,41 days 02:02:54.671036180,4.847302,5.869990


#### For some recordings the age at the time of transfer is "negative"  - these need to be corrected

In [57]:
clin_df[clin_df['Postnatal Age'] < pd.to_timedelta(0)]

,Case ID,Date of Birth,Gestational Age,Birth Weight,Actual Weight,Pathology,Start,End,Weight,Recording start,Recording end,Duration,Postnatal Age,Corrected gestational Age,Gestational Age (weeks),Corrected gestational Age (weeks)


#### For some recordings the duration of the recording is "negative"  - these need to be corrected

In [60]:
clin_df[clin_df['Duration'] < pd.to_timedelta(0)]

,Case ID,Date of Birth,Gestational Age,Birth Weight,Actual Weight,Pathology,Start,End,Weight,Recording start,Recording end,Duration,Postnatal Age,Corrected gestational Age,Gestational Age (weeks),Corrected gestational Age (weeks)


#### Babies was at less than 23 weeks gestation

In [63]:
clin_df[clin_df['Gestational Age (weeks)'] < 23]

,Case ID,Date of Birth,Gestational Age,Birth Weight,Actual Weight,Pathology,Start,End,Weight,Recording start,Recording end,Duration,Postnatal Age,Corrected gestational Age,Gestational Age (weeks),Corrected gestational Age (weeks)
AL000294,47588,2018-07-10 11:30:00,147 days,300,,"[Neonat immat gr s 21 (P07.2) , Resuscitatio...","Tue, 10 Jul 2018 122935 +0200","Tue, 10 Jul 2018 131103 +0200",300,2018-07-10 12:29:35,2018-07-10 13:11:03,0 days 00:41:28,0 days 01:41:03,147 days 01:41:03,21.0,21.0
AL000561,50979,2019-06-01 22:08:00,154 days,580,,"[Neonat immat gr s 22 (P07.2) , Gemini A (P0...","Sat, 01 Jun 2019 224754 +0200","Sun, 02 Jun 2019 011426 +0200",580,2019-06-01 22:47:54,2019-06-02 01:14:26,0 days 02:26:32,0 days 03:06:26,154 days 03:06:26,22.0,22.0


#### Babies born with less than 500 g birth weight

In [66]:
clin_df[clin_df['Birth Weight'] < 500]

,Case ID,Date of Birth,Gestational Age,Birth Weight,Actual Weight,Pathology,Start,End,Weight,Recording start,Recording end,Duration,Postnatal Age,Corrected gestational Age,Gestational Age (weeks),Corrected gestational Age (weeks)
AL000038,42889,2017-04-01 00:00:00,161 days,490,690,"[Neonat immat gr s 22 (P07.2) 22-23 , Ileus (...","Sat, 29 Apr 2017 091252 +0200","Sat, 29 Apr 2017 105944 +0200",690,2017-04-29 09:12:52,2017-04-29 10:59:44,0 days 01:46:52,28 days 10:59:44,189 days 10:59:44,23.0,27.1
AL000066,43365,2017-05-18 14:24:00,168 days,450,650,"[PDA (Q25.0) , Neonat immat gr s 24 (P07.2) ...","Wed, 14 Jun 2017 105336 +0200","Wed, 14 Jun 2017 121541 +0200",650,2017-06-14 10:53:36,2017-06-14 12:15:41,0 days 01:22:05,26 days 21:51:41,194 days 21:51:41,24.0,27.8
AL000099,43778,2017-06-10 10:44:00,182 days,490,830,"[Neonat immat gr s 26 (P07.2) , NEC (P77) ,...","Sun, 23 Jul 2017 151033 +0200","Sun, 23 Jul 2017 155506 +0200",830,2017-07-23 15:10:33,2017-07-23 15:55:06,0 days 00:44:33,43 days 05:11:06,225 days 05:11:06,26.0,32.2
AL000100,43779,2017-04-01 00:00:00,175 days,490,1890,"[ROP (H35.1) , BPD (P27.1) ]","Mon, 24 Jul 2017 083540 +0200","Mon, 24 Jul 2017 091856 +0200",1890,2017-07-24 08:35:40,2017-07-24 09:18:56,0 days 00:43:16,114 days 09:18:56,289 days 09:18:56,25.0,41.3
AL000101,43781,2017-04-01 00:00:00,175 days,490,1890,"[ROP (H35.1) st.p. vitrectomian l.u. , BPD (P...","Mon, 24 Jul 2017 110810 +0200","Mon, 24 Jul 2017 114954 +0200",1890,2017-07-24 11:08:10,2017-07-24 11:49:54,0 days 00:41:44,114 days 11:49:54,289 days 11:49:54,25.0,41.4
AL000140,44200,2017-08-08 10:20:00,161 days,410,640,"[Neonat immat gr s 23 (P07.2) , NEC (P77) ,...","Fri, 25 Aug 2017 132348 +0200","Fri, 25 Aug 2017 151422 +0200",640,2017-08-25 13:23:48,2017-08-25 15:14:22,0 days 01:50:34,17 days 04:54:22,178 days 04:54:22,23.0,25.5
AL000163,45238,2017-07-13 18:07:00,168 days,360,2880,"[BPD (P27.1) , Légzési elégtelenség (P28.5) ...","Sat, 25 Nov 2017 144108 +0100","Sat, 25 Nov 2017 164239 +0100",2880,2017-11-25 14:41:08,2017-11-25 16:42:39,0 days 02:01:31,134 days 22:35:39,302 days 22:35:39,24.0,43.3
AL000182,45454,2017-07-13 23:05:00,168 days,360,3300,"[Neonat immat gr s 24 (P07.2) , Hernia ingui...","Wed, 13 Dec 2017 124334 +0100","Wed, 13 Dec 2017 144535 +0100",3300,2017-12-13 12:43:34,2017-12-13 14:45:35,0 days 02:02:01,152 days 15:40:35,320 days 15:40:35,24.0,45.8
AL000294,47588,2018-07-10 11:30:00,147 days,300,,"[Neonat immat gr s 21 (P07.2) , Resuscitatio...","Tue, 10 Jul 2018 122935 +0200","Tue, 10 Jul 2018 131103 +0200",300,2018-07-10 12:29:35,2018-07-10 13:11:03,0 days 00:41:28,0 days 01:41:03,147 days 01:41:03,21.0,21.0
AL000348,48173,2018-07-27 00:00:00,175 days,490,,"[Neonat immat gr s 25 (P07.2) , Hypothermia ...","Tue, 28 Aug 2018 153653 +0200","Tue, 28 Aug 2018 162146 +0200",490,2018-08-28 15:36:53,2018-08-28 16:21:46,0 days 00:44:53,32 days 16:21:46,207 days 16:21:46,25.0,29.7


In [68]:
len(clin_df[clin_df['Birth Weight'] < 500])

13

#### Babies transferred with the postnatal age of > 46 weeks we need to discuss whether to include them in the data analysis

In [71]:
a = clin_df[clin_df['Corrected gestational Age (weeks)'] > 46]
a.sort_values('Corrected gestational Age (weeks)')

,Case ID,Date of Birth,Gestational Age,Birth Weight,Actual Weight,Pathology,Start,End,Weight,Recording start,Recording end,Duration,Postnatal Age,Corrected gestational Age,Gestational Age (weeks),Corrected gestational Age (weeks)
AL000451,49876,2019-01-02 00:00:00,287 days,3000,3720,"[Stenosis tracheae (Q32.1) , Infectio (P39.9...","Thu, 07 Feb 2019 063812 +0100","Thu, 07 Feb 2019 070516 +0100",3720,2019-02-07 06:38:12,2019-02-07 07:05:16,0 days 00:27:04,36 days 07:05:16,323 days 07:05:16,41.0,46.2
AL000782,53788,2019-12-15 10:07:00,266 days,4020,4230,"[Neonat mat gr s 38 (P96.4) , Vitium cordis ...","Tue, 11 Feb 2020 102153 +0100","Tue, 11 Feb 2020 113111 +0100",4230,2020-02-11 10:21:53,2020-02-11 11:31:11,0 days 01:09:18,58 days 01:24:11,324 days 01:24:11,38.0,46.3
AL000823,54127,2020-01-14 00:00:00,266 days,3580,,"[Neonat mat gr s 38 (P96.4) , ASD (Q21.1) ,...","Fri, 13 Mar 2020 155505 +0100","Fri, 13 Mar 2020 171936 +0100",3580,2020-03-13 15:55:05,2020-03-13 17:19:36,0 days 01:24:31,59 days 17:19:36,325 days 17:19:36,38.0,46.5
AL000459,49912,2019-01-02 00:00:00,287 days,3000,3720,"[Foramen ovale apertum (Q21.1) , Stenosis tr...","Mon, 11 Feb 2019 120638 +0100","Mon, 11 Feb 2019 124448 +0100",3720,2019-02-11 12:06:38,2019-02-11 12:44:48,0 days 00:38:10,40 days 12:44:48,327 days 12:44:48,41.0,46.8
AL000688,52223,2019-07-07 09:05:00,259 days,2265,4245,"[Neonat mat gr s 37 (P96.4) , Postasphyxias ...","Wed, 18 Sep 2019 090008 +0200","Wed, 18 Sep 2019 092730 +0200",4245,2019-09-18 09:00:08,2019-09-18 09:27:30,0 days 00:27:22,73 days 00:22:30,332 days 00:22:30,37.0,47.4
AL000819,54105,2019-10-14 00:00:00,182 days,950,3250,"[Pneumonia (J18.9) , Légzési elégtelenség (P...","Thu, 12 Mar 2020 103614 +0100","Thu, 12 Mar 2020 112302 +0100",3250,2020-03-12 10:36:14,2020-03-12 11:23:02,0 days 00:46:48,150 days 11:23:02,332 days 11:23:02,26.0,47.5
AL000042,42922,2017-02-23 00:00:00,266 days,3000,4900,"[Neonat mat gr s 39 (P96.4) , Convulsio (P90...","Tue, 02 May 2017 193707 +0200","Tue, 02 May 2017 202154 +0200",4900,2017-05-02 19:37:07,2017-05-02 20:21:54,0 days 00:44:47,68 days 20:21:54,334 days 20:21:54,38.0,47.8
AL000043,42923,2017-02-23 00:00:00,266 days,3000,4900,"[Neonat mat gr s 38 (P96.4) , Convulsio (P90...","Tue, 02 May 2017 204804 +0200","Tue, 02 May 2017 211132 +0200",4900,2017-05-02 20:48:04,2017-05-02 21:11:32,0 days 00:23:28,68 days 21:11:32,334 days 21:11:32,38.0,47.8
AL000478,50349,2019-02-01 00:00:00,280 days,3550,4310,"[Hypotonia musculorum (P94.2) , Légzészavar ...","Wed, 27 Mar 2019 123644 +0100","Wed, 27 Mar 2019 131024 +0100",4310,2019-03-27 12:36:44,2019-03-27 13:10:24,0 days 00:33:40,54 days 13:10:24,334 days 13:10:24,40.0,47.8
AL000690,52261,2019-07-07 09:05:00,259 days,2265,4200,"[Encephalopathia (G93.4) postasphyxias , Gast...","Sun, 22 Sep 2019 051336 +0200","Sun, 22 Sep 2019 063040 +0200",4200,2019-09-22 05:13:36,2019-09-22 06:30:40,0 days 01:17:04,76 days 21:25:40,335 days 21:25:40,37.0,48.0


In [73]:
len(clin_df[clin_df['Corrected gestational Age (weeks)'] > 46])

37

### 9. Identify ICD codes

In [76]:
icd = re.compile(r'\(\S\d+\.?\d*\)')

In [78]:
def icd_finder(lst):
    icd_list = []
    for item in lst:
        if icd.findall(item):
            icd_list.append(icd.findall(item)[0])
    
    return icd_list

In [80]:
def icd_cleanup(lst):
    icd_list = []
    for item in lst:
        if '.' in item:
            new_item = item[0 : item.index('.')] + item[item.index('.') + 1 : ]
        else:
            new_item = item
        icd_list.append(new_item[1:-1])
    return icd_list

In [82]:
def icd_cleanup_2(lst):
    # Some BNO codes have 4 digits and end with zero. These are the same codes as the same ones with 3 digits (without zero at the end)
    icd_list = []
    for item in lst:
        if len(item) == 5 and item.endswith('0'):
            icd_list.append(item[:-1])
        else:
            icd_list.append(item)
    icd_lst = sorted(set(icd_list))
    return icd_list

In [84]:
clin_df['ICD'] = clin_df['Pathology'].apply(icd_finder)
clin_df['ICD'] = clin_df['ICD'].apply(icd_cleanup)
clin_df['ICD'] = clin_df['ICD'].apply(icd_cleanup_2)
clin_df['ICD'];

### 10. Identify all icd codes

In [87]:
icd_all = []
for _, item in clin_df.iterrows():
    icd_all.extend(item['ICD'])    
icd_all = sorted(set(icd_all))
len(icd_all)

193

In [89]:
icd_all_frme = DataFrame(icd_all)
icd_all_frme.columns = ['code']
icd_all_frme.sort_values(by = 'code', inplace = True)
len(icd_all_frme)

193

In [91]:
writer = pd.ExcelWriter(os.path.join(DIR_WRITE, 'icd_codes.xlsx'))
icd_all_frme.to_excel(writer, 'icd_codes')
writer.close()

### 11. Import the file that has been manually curated from the exported `icd_codes.xlsx` files to contain all relevant diagnosis previously identified

In [93]:
icd_codes = pd.read_excel(os.path.join(PATH, 'ventilation_fabian', 'icd_codes_curated_new.xlsx'), 
    usecols = [0,1], index_col = 0)

Identify new codes not in the dataset so far

In [97]:
new_diagnoses = sorted(set(icd_all_frme['code']) - set(icd_codes.index))
new_diagnoses

['G039', 'Q393']

At this point the `icd_codes_curated_new.xlsx` file needs to be manually curated with these entries.

### 12. Import the now curated `icd_codes.xlsx` files to contain now all relevant diagnosis including new ones

In [124]:
icd_codes = pd.read_excel(os.path.join(PATH, 'ventilation_fabian', 'icd_codes_curated_new.xlsx'), 
    usecols = [0,1], index_col = 0)

In [126]:
# Now there are no new codes
sorted(set(icd_all_frme['code']) - set(icd_codes.index))

[]

### 13. Create Pathology column with English names

In [129]:
icd_dictionary = dict(zip(icd_codes.index, icd_codes['name']))

In [131]:
def icd_replace(lst):
    icd_list = []
    for item in lst:
        new_item = icd_dictionary[item]
        icd_list.append(new_item)
    return icd_list

In [133]:
clin_df['Pathology_English'] = clin_df['ICD'].apply(icd_replace)

In [135]:
clin_df.tail()

,Case ID,Date of Birth,Gestational Age,Birth Weight,Actual Weight,Pathology,Start,End,Weight,Recording start,Recording end,Duration,Postnatal Age,Corrected gestational Age,Gestational Age (weeks),Corrected gestational Age (weeks),ICD,Pathology_English
AL001043,55312,2020-08-13 09:29:00,273 days,3300,3356,"[Neonat mat gr s 39 (P96.4) , Asphyxia perin...","Tue, 18 Aug 2020 123637 +0200","Tue, 18 Aug 2020 133138 +0200",3356,2020-08-18 12:36:37,2020-08-18 13:31:38,0 days 00:55:01,5 days 04:02:38,278 days 04:02:38,39.0,39.7,"[P964, P219, P528, _000]","[Term newborn, Perinatal asphyxia, Other intra..."
AL001047,55343,2020-08-12 00:00:00,182 days,840,755,"[Neonat immat gr s 26 (P07.2) , RDS (P22) ,...","Fri, 21 Aug 2020 215302 +0200","Fri, 21 Aug 2020 224520 +0200",755,2020-08-21 21:53:02,2020-08-21 22:45:20,0 days 00:52:18,9 days 22:45:20,191 days 22:45:20,26.0,27.4,"[P072, P22, P77, K631, P708, P369, _000]","[Extreme immaturity of newborn, unspecified we..."
AL001054,55383,2020-08-26 09:45:00,273 days,3650,,"[Neonat mat gr s 39 (P96.4) , Infectio (P39....","Wed, 26 Aug 2020 121824 +0200","Wed, 26 Aug 2020 130420 +0200",3650,2020-08-26 12:18:24,2020-08-26 13:04:20,0 days 00:45:56,0 days 03:19:20,273 days 03:19:20,39.0,39.0,"[P964, P399, _000]","[Term newborn, Infection specific to the perin..."
AL001055,55393,2020-06-17 15:50:00,196 days,1246,3250,"[Neonat praemat gr s 28 (P07.3) , BPD (P27.1...","Thu, 27 Aug 2020 110758 +0200","Thu, 27 Aug 2020 131956 +0200",3250,2020-08-27 11:07:58,2020-08-27 13:19:56,0 days 02:11:58,70 days 21:29:56,266 days 21:29:56,28.0,38.1,"[P073, P271, I270, K409, _000, P399]","[Preterm newborn, unspecified weeks of gestati..."
AL001086,55607,2020-09-04 17:56:00,252 days,2860,3160,"[Neonat praemat gr s 36 (P07.3) , Légzési el...","Mon, 21 Sep 2020 143058 +0200","Mon, 21 Sep 2020 151920 +0200",3160,2020-09-21 14:30:58,2020-09-21 15:19:20,0 days 00:48:22,16 days 21:23:20,268 days 21:23:20,36.0,38.4,"[P073, P285, _000, Z930, Q288, D489]","[Preterm newborn, unspecified weeks of gestati..."


### 14. Final cleanup of the DataFrame

In [138]:
clin_df.columns

Index(['Case ID', 'Date of Birth', 'Gestational Age', 'Birth Weight',
       'Actual Weight', 'Pathology', 'Start', 'End', 'Weight',
       'Recording start', 'Recording end', 'Duration', 'Postnatal Age',
       'Corrected gestational Age', 'Gestational Age (weeks)',
       'Corrected gestational Age (weeks)', 'ICD', 'Pathology_English'],
      dtype='object')

In [140]:
column_list = ['Case ID', 'Date of Birth', 'Gestational Age (weeks)', 'Birth Weight',
              'Postnatal Age', 'Corrected gestational Age (weeks)',  'Weight',
              'ICD', 'Pathology_English', 'Recording start', 'Recording end', 'Duration',]     
clin_df = clin_df[column_list]

In [142]:
clin_df.head()

,Case ID,Date of Birth,Gestational Age (weeks),Birth Weight,Postnatal Age,Corrected gestational Age (weeks),Weight,ICD,Pathology_English,Recording start,Recording end,Duration
AL000003,42543,2017-03-24 17:41:00,28.0,990,0 days 02:09:04,28.0,990,"[P073, P220, P704]","[Preterm newborn, unspecified weeks of gestati...",2017-03-24 18:05:07,2017-03-24 19:50:04,0 days 01:44:57
AL000005,42552,2017-03-26 17:50:00,37.0,3530,0 days 02:07:19,37.0,3530,"[P964, P399, P228]","[Term newborn, Infection specific to the perin...",2017-03-26 19:14:53,2017-03-26 19:57:19,0 days 00:42:26
AL000006,42554,2017-03-26 23:37:00,31.0,1470,0 days 01:18:07,31.0,1470,"[P073, P015, Q792, Q205]","[Preterm newborn, unspecified weeks of gestati...",2017-03-27 00:04:21,2017-03-27 00:55:07,0 days 00:50:46
AL000007,42578,2017-01-29 00:00:00,40.0,3200,59 days 02:48:57,48.4,4800,"[E86, P285, I500, Z518, P964, E141]","[Dehydration, Respiratory failure of newborn, ...",2017-03-29 00:48:29,2017-03-29 02:48:57,0 days 02:00:28
AL000008,42596,2017-03-29 13:20:00,38.0,3230,0 days 04:22:48,38.0,3230,"[P964, Z381, P219, P90, P708, P809]","[Term newborn, Single liveborn infant, born ou...",2017-03-29 15:44:26,2017-03-29 17:42:48,0 days 01:58:22


### 15. Statistics on clinical data

In [145]:
clinical_stats = round(clin_df.describe(percentiles = [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]), 1)
clinical_stats

,Date of Birth,Gestational Age (weeks),Birth Weight,Postnatal Age,Corrected gestational Age (weeks),Weight,Recording start,Recording end,Duration
count,819,819.0,819.0,819,819.0,819.0,819,819,819
mean,2019-02-14 23:58:38.534798592,34.7,2433.5,12 days 19:44:30.918192918,36.5,2600.5,2019-02-27 18:27:50.815628544,2019-02-27 19:43:09.452991232,0 days 01:15:18.637362637
min,2016-12-13 00:00:00,21.0,300.0,0 days 00:01:09,21.0,300.0,2017-03-24 18:05:07,2017-03-24 19:50:04,0 days 00:05:44
1%,2017-02-23 08:38:24,23.0,490.0,0 days 01:09:37.820000,24.0,563.6,2017-04-03 19:55:43.880000,2017-04-03 20:47:58.860000,0 days 00:09:13.760000
5%,2017-04-27 22:32:18,25.0,700.0,0 days 01:37:17.900000,27.1,830.0,2017-05-16 03:22:12.600000,2017-05-16 04:38:42.400000,0 days 00:25:11
25%,2018-04-12 00:00:00,32.0,1690.0,0 days 03:08:40,33.0,1840.0,2018-04-14 20:56:54.500000,2018-04-14 22:05:06.500000,0 days 00:45:08
50%,2019-04-17 00:00:00,36.0,2600.0,0 days 07:03:47,37.0,2700.0,2019-04-28 16:18:46,2019-04-28 18:40:14,0 days 01:05:40
75%,2020-02-07 05:34:30,38.0,3195.0,6 days 06:45:28,39.1,3325.0,2020-02-20 13:55:56.500000,2020-02-20 14:58:44.500000,0 days 01:36:41
95%,2020-07-14 13:40:12,40.0,3900.0,72 days 02:51:57.599999992,45.8,4210.5,2020-07-22 16:11:02.400000,2020-07-22 16:40:58.700000,0 days 02:22:49.300000
99%,2020-08-13 09:29:00,41.0,4458.2,137 days 02:08:28.139999996,55.1,4900.0,2020-08-16 19:59:37.700000,2020-08-16 21:37:09.040000,0 days 03:14:01.219999999


### 16. Export clinical information data in tables and individual text files

##### Create sub-directories for each case if it does not yet exist

In [149]:
# Images and raw data will be written on an external hard drive
os.makedirs(os.path.join(DATA_DUMP, 'fabian_cases'), exist_ok = True)

for case in sorted(clin_df.index): 
    os.makedirs(os.path.join(DATA_DUMP, 'fabian_cases', case), exist_ok = True)

##### Export clinical data about individual cases as text files

In [152]:
# Clinical info about all recordings which clinical data are available and are over 15 minutes long

for case in sorted(clin_df.index):
    
    fileout = open(os.path.join(DATA_DUMP, 'fabian_cases', case, f'{case}_clin_info.txt'), 'w')
    
    fileout.write('Case ID:             %-50s\n' % case)
    fileout.write('Start:               %-50s\n' % datetime.strftime(clin_df.loc[case]['Recording start'], '%d/%m/%Y %H:%M:%S', ))
    fileout.write('End:                 %-50s\n' % datetime.strftime(clin_df.loc[case]['Recording end'], '%d/%m/%Y %H:%M:%S', ))
    fileout.write('Duration:            %-50s\n' % clin_df.loc[case]['Duration'])
    fileout.write('Gestational age:     %-50s\n' % clin_df.loc[case]['Gestational Age (weeks)'])
    fileout.write('Postmenstrual age:   %-50s\n' % clin_df.loc[case]['Corrected gestational Age (weeks)'])
    fileout.write('Birth Weight:        %-50s\n' % clin_df.loc[case]['Birth Weight'])
    fileout.write('Weight:              %-50s\n' % clin_df.loc[case]['Weight'])
    fileout.write('ICD:                 %-50s\n' % ', '.join(clin_df.loc[case]['ICD']))
    fileout.write('Diagnoses:           %-50s\n' % ', '.join(clin_df.loc[case]['Pathology_English']))
    
    fileout.close()

##### Export clinical information as an Excel sheet

In [248]:
writer = pd.ExcelWriter(os.path.join(DIR_WRITE, 'clinical_data_all_1_1100.xlsx'))
clin_df.to_excel(writer, 'clin_df')
writer.close()

### 17. Export statistics on clinical data

In [251]:
writer = pd.ExcelWriter(os.path.join(DIR_WRITE, 'clinical_stats_1_1100.xlsx'))
clinical_stats.to_excel(writer, 'stats')
writer.close()

### 18. Export processed data as pickle files

In [160]:
with open(os.path.join(DATA_DUMP, 'clin_df_1_1100.pickle'), 'wb') as handle:
    pickle.dump(clin_df, handle, protocol=pickle.HIGHEST_PROTOCOL)

### 19. Create patient lists for various disease groups

#### A. RDS

In [165]:
RDS_dg = {'P22', 'P220'}

In [167]:
RDS_cases = []
for case, dgs in clin_df['ICD'].items():
    if set(dgs).intersection(RDS_dg):
        RDS_cases.append(case)

In [169]:
clin_df_RDS = clin_df.loc[RDS_cases]
clin_df_RDS;

In [171]:
len(clin_df_RDS)

256

#### B. HIE

In [174]:
HIE_dg = ['P219', 'Z518', 'Z548',]  

In [176]:
HIE_cases = []
for case, dgs in clin_df['ICD'].items():
    if set(dgs).intersection(HIE_dg):
        HIE_cases.append(case)

In [178]:
clin_df_HIE = clin_df.loc[HIE_cases]
clin_df_HIE;

In [180]:
len(clin_df_HIE)

151

#### C. Meconium aspiration

In [183]:
MAS_dg = ['P240',]

In [185]:
MAS_cases = []
for case, dgs in clin_df['ICD'].items():
    if set(dgs).intersection(MAS_dg):
        MAS_cases.append(case)

In [187]:
clin_df_MAS = clin_df.loc[MAS_cases]
clin_df_MAS;

In [189]:
len(clin_df_MAS)

58

#### D. PPHN

In [192]:
PPHN_dg = ['P293', ]

In [194]:
PPHN_cases = []
for case, dgs in clin_df['ICD'].items():
    if set(dgs).intersection(PPHN_dg):
        PPHN_cases.append(case)

In [196]:
clin_df_PPHN = clin_df.loc[PPHN_cases]
clin_df_PPHN;

In [198]:
len(clin_df_PPHN)

43

#### E. Congenital diaphragmatic hernia

In [201]:
CDH_dg = ['Q790', 'Q791', 'K449']

In [203]:
CDH_cases = []
for case, dgs in clin_df['ICD'].items():
    if set(dgs).intersection(CDH_dg):
        CDH_cases.append(case)

In [205]:
clin_df_CDH = clin_df.loc[CDH_cases]
clin_df_CDH;

In [207]:
len(clin_df_CDH)

15

#### F. Necrotizing enterocolitis (NEC)

In [210]:
NEC_dg = ['P77',]

In [212]:
NEC_cases = []
for case, dgs in clin_df['ICD'].items():
    if set(dgs).intersection(NEC_dg):
        NEC_cases.append(case)

In [214]:
clin_df_NEC = clin_df.loc[NEC_cases]
clin_df_NEC;

In [216]:
len(clin_df_NEC)

35

#### G. Surgical cases (except NEC, CDH and cardiac)

In [219]:
surgical_dg = ['K409', 'K449', 'K562', 'K566', 'K631', 'K9210', 'Q059', 'Q321', 'Q391', 'Q392', 'Q423' , 'Q431',
               'Q438', 'Q4380', 'Q549', 'Q556', 'Q641', 'Q792', 'Q793', 'R1000']

In [221]:
surgical_cases = []
for case, dgs in clin_df['ICD'].items():
    if set(dgs).intersection(surgical_dg):
        surgical_cases.append(case)

In [223]:
clin_df_surgical = clin_df.loc[surgical_cases]
clin_df_surgical;

In [225]:
len(clin_df_surgical)

55

#### H. Cardiac cases (except PFO/ASD2 and PDA)

Exclude:
- Q211:	Atrial septal defect
- Q250	Patent ductus arteriosus

In [228]:
cardiac_dg = ['I270', 'I30', 'I340', 'I342', 'I421', 'I429', 'I442', 'I443', 'I471', 'I472', 'I479', 'I500', 'I509', 'I514', 'I517',
              'Q200', 'Q201', 'Q203', 'Q205', 'Q209', 'Q210', 'Q212', 'Q213', 'Q220', 'Q221', 'Q223', 'Q224', 'Q225', 'Q228', 'Q230', 
              'Q232', 'Q234', 'Q240', 'Q244', 'Q245', 'Q251', 'Q252', 'Q253', 'Q254', 'Q262', 'Q273', 'Q279']

In [230]:
cardiac_cases = []
for case, dgs in clin_df['ICD'].items():
    if set(dgs).intersection(cardiac_dg):
        cardiac_cases.append(case)

In [232]:
clin_df_cardiac = clin_df.loc[cardiac_cases]
clin_df_cardiac;

In [234]:
len(clin_df_cardiac)

132

### 20. Export clinical dataframes into a multisheet Excel file

In [254]:
writer = pd.ExcelWriter(os.path.join(DIR_WRITE, 'clinical_data_diseases_1_1100.xlsx'))
clin_df.to_excel(writer, 'all')
clin_df_cardiac.to_excel(writer, 'cardiac')
clin_df_CDH.to_excel(writer, 'CDH')
clin_df_HIE.to_excel(writer, 'HIE')
clin_df_MAS.to_excel(writer, 'MAS')
clin_df_NEC.to_excel(writer, 'NEC')
clin_df_PPHN.to_excel(writer, 'PPHN')
clin_df_RDS.to_excel(writer, 'RDS')
clin_df_surgical.to_excel(writer, 'surgical')
writer.close()

### 21. Export selected clinical data as pickle archive

In [239]:
with open(os.path.join(DATA_DUMP, 'clin_df_cardiac.pickle'), 'wb') as handle:
    pickle.dump(clin_df_cardiac, handle, protocol=pickle.HIGHEST_PROTOCOL)   
with open(os.path.join(DATA_DUMP, 'clin_df_CDH.pickle'), 'wb') as handle:
    pickle.dump(clin_df_CDH, handle, protocol=pickle.HIGHEST_PROTOCOL)
with open(os.path.join(DATA_DUMP, 'clin_df_HIE.pickle'), 'wb') as handle:
    pickle.dump(clin_df_HIE, handle, protocol=pickle.HIGHEST_PROTOCOL)   
with open(os.path.join(DATA_DUMP, 'clin_df_MAS.pickle'), 'wb') as handle:
    pickle.dump(clin_df_MAS, handle, protocol=pickle.HIGHEST_PROTOCOL)
with open(os.path.join(DATA_DUMP, 'clin_df_NEC.pickle'), 'wb') as handle:
    pickle.dump(clin_df_NEC, handle, protocol=pickle.HIGHEST_PROTOCOL)   
with open(os.path.join(DATA_DUMP, 'clin_df_PPHN.pickle'), 'wb') as handle:
    pickle.dump(clin_df_PPHN, handle, protocol=pickle.HIGHEST_PROTOCOL)
with open(os.path.join(DATA_DUMP, 'clin_df_RDS.pickle'), 'wb') as handle:
    pickle.dump(clin_df_RDS, handle, protocol=pickle.HIGHEST_PROTOCOL)   
with open(os.path.join(DATA_DUMP, 'clin_df_surgical.pickle'), 'wb') as handle:
    pickle.dump(clin_df_surgical, handle, protocol=pickle.HIGHEST_PROTOCOL)      